In [19]:
import sympy as sp

x = sp.symbols("x")

In [20]:
def trimf(a, b, c):
    return sp.Piecewise(
        (0, x <= a),
        ((x - a) / (b - a), x <= b),
        ((c - x) / (c - b), x < c),
        (0, True)
    )

In [21]:
def trapmf(a, b, c, d):
    if a == b:
        return sp.Piecewise(
            (0, x < a),
            (1, x <= c),
            ((d - x) / (d - c), x < d),
            (0, True)
        )

    if c == d:
        return sp.Piecewise(
            (0, x <= a),
            ((x - a) / (b - a), x < b),
            (1, x <= d),
            (0, True)
        )

    return sp.Piecewise(
        (0, x <= a),
        ((x - a) / (b - a), x < b),
        (1, x <= c),
        ((d - x) / (d - c), x < d),
        (0, True)
    )


In [22]:
def degree(expr, nilai):
    return float(expr.subs(x, nilai))

In [23]:
def input_range(teks, minimum, maksimum):
    while True:
        try:
            nilai = float(input(teks))
            if minimum <= nilai <= maksimum:
                return nilai
            print(f"Input harus dalam range {minimum}-{maksimum}.")
        except ValueError:
            print("Input harus berupa angka.")


In [24]:
def kategori_tip(nilai):
    if nilai < 10:
        return "Rendah"
    if nilai < 15:
        return "Sedang"
    if nilai <= 20:
        return "Tinggi"
    return "Sangat Tinggi"

In [25]:
makanan_input = input_range("Masukkan kualitas makanan (0-10): ", 0, 10)
pelayanan_input = input_range("Masukkan kualitas pelayanan (0-10): ", 0, 10)

In [26]:
makanan_mf = {
    "Buruk": trapmf(0, 0, 2, 4),
    "Biasa": trimf(3, 5, 7),
    "Enak": trapmf(6, 8, 10, 10)
}

pelayanan_mf = {
    "Buruk": trapmf(0, 0, 2, 4),
    "Biasa": trimf(3, 5, 7),
    "Baik": trapmf(6, 8, 10, 10)
}

In [27]:
makanan = {k: degree(v, makanan_input) for k, v in makanan_mf.items()}
pelayanan = {k: degree(v, pelayanan_input) for k, v in pelayanan_mf.items()}

In [28]:
rules = [
    ("R1", "Buruk", "Buruk", 5),
    ("R2", "Buruk", "Biasa", 8),
    ("R3", "Buruk", "Baik", 12),
    ("R4", "Biasa", "Buruk", 8),
    ("R5", "Biasa", "Biasa", 15),
    ("R6", "Biasa", "Baik", 18),
    ("R7", "Enak", "Buruk", 12),
    ("R8", "Enak", "Biasa", 18),
    ("R9", "Enak", "Baik", 25)
]

In [29]:
evaluasi = []
for kode, m, p, z in rules:
    w = min(makanan[m], pelayanan[p])
    evaluasi.append((kode, w, z))

numerator = sum(w * z for _, w, z in evaluasi)
denominator = sum(w for _, w, _ in evaluasi)
tip = numerator / denominator if denominator != 0 else 0.0
aktif = [kode for kode, w, _ in evaluasi if w > 0]

In [30]:
print("INPUT:")
print(f"Kualitas Makanan: {makanan_input:.2f} / 10")
print(f"Kualitas Pelayanan: {pelayanan_input:.2f} / 10")

INPUT:
Kualitas Makanan: 7.50 / 10
Kualitas Pelayanan: 4.00 / 10


In [31]:
print("\n----------------------------------------")
print("FUZZIFIKASI")
print("----------------------------------------")
print("Makanan:")
print(f"Buruk: {makanan['Buruk']:.6f}")
print(f"Biasa: {makanan['Biasa']:.6f}")
print(f"Enak: {makanan['Enak']:.6f}")


----------------------------------------
FUZZIFIKASI
----------------------------------------
Makanan:
Buruk: 0.000000
Biasa: 0.000000
Enak: 0.750000


In [32]:
print("Pelayanan:")
print(f"Buruk: {pelayanan['Buruk']:.6f}")
print(f"Biasa: {pelayanan['Biasa']:.6f}")
print(f"Baik: {pelayanan['Baik']:.6f}")

Pelayanan:
Buruk: 0.000000
Biasa: 0.500000
Baik: 0.000000


In [33]:
print("\n----------------------------------------")
print("EVALUASI ATURAN")
print("----------------------------------------")
for i in range(0, len(evaluasi), 2):
    kiri = evaluasi[i]
    if i + 1 < len(evaluasi):
        kanan = evaluasi[i + 1]
        print(f"{kiri[0]}: w={kiri[1]:.6f}, z={kiri[2]} | {kanan[0]}: w={kanan[1]:.6f}, z={kanan[2]}")
    else:
        print(f"{kiri[0]}: w={kiri[1]:.6f}, z={kiri[2]}")


----------------------------------------
EVALUASI ATURAN
----------------------------------------
R1: w=0.000000, z=5 | R2: w=0.000000, z=8
R3: w=0.000000, z=12 | R4: w=0.000000, z=8
R5: w=0.000000, z=15 | R6: w=0.000000, z=18
R7: w=0.000000, z=12 | R8: w=0.500000, z=18
R9: w=0.000000, z=25


In [34]:
print("\n----------------------------------------")
print("DEFUZZIFIKASI SUGENO")
print("----------------------------------------")
print(f"Numerator: Σ(wi×zi) = {numerator:.6f}")
print(f"Denominator: Σ(wi) = {denominator:.6f}")
print(f"Tip = {tip:.2f}%")
print(f"Kategori: {kategori_tip(tip)}")


----------------------------------------
DEFUZZIFIKASI SUGENO
----------------------------------------
Numerator: Σ(wi×zi) = 9.000000
Denominator: Σ(wi) = 0.500000
Tip = 18.00%
Kategori: Tinggi


In [35]:
print("\n----------------------------------------")
print("TOLAK UKUR VALIDASI:")
print("----------------------------------------")
print(f"μ_Enak({makanan_input:.1f}) = ({makanan_input:.1f}-6)/(8-6) = {makanan['Enak']:.2f}")
print(
    f"μ_Biasa_pelayanan({pelayanan_input:.1f}) = ({pelayanan_input:.0f}-3)/(5-3) = {pelayanan['Biasa']:.2f} WAIT - recalculate:")
print(
    f"Actually: pelayanan={pelayanan_input:.1f}, triangular(3,5,7): ({pelayanan_input:.0f}-3)/(5-3) = {pelayanan['Biasa']:.2f}")
print(f"Koreksi: μ_Biasa_pelayanan({pelayanan_input:.1f}) = {pelayanan['Biasa']:.2f}, bukan 1.0")
print(
    f"R8: min({makanan['Enak']:.2f}, {pelayanan['Biasa']:.2f}) = {evaluasi[7][1]:.2f}, R5: min({makanan['Biasa']:.0f}, {pelayanan['Biasa']:.2f}) = {evaluasi[4][1]:.0f}")
print(
    f"Juga R4: min({makanan['Biasa']:.0f}, {pelayanan['Buruk']:.0f}) = {evaluasi[3][1]:.0f}, tapi pelayanan Buruk({pelayanan_input:.0f}) = {pelayanan['Buruk']:.0f} karena {pelayanan_input:.0f}>=4 - 0")

print("\nRecalculate properly:")
print(f"Pelayanan={pelayanan_input:.1f}:")
print(f"Buruk: trapezoidal(0,0,2,4) - ({pelayanan_input:.0f}-4)/(4-2) = {pelayanan['Buruk']:.1f} (x>=d - 0)")
print(f"Biasa: triangular(3,5,7) - ({pelayanan_input:.0f}-3)/(5-3) = {pelayanan['Biasa']:.1f}")
print(f"Baik: trapezoidal(6,8,10,10) - {pelayanan['Baik']:.0f}")

print(f"\nMakanan={makanan_input:.1f}:")
print(f"Buruk: trapezoidal(0,0,2,4) - {makanan['Buruk']:.0f}")
print(f"Biasa: triangular(3,5,7) - {makanan_input:.1f}>=7 - {makanan['Biasa']:.0f}")
print(f"Enak: trapezoidal(6,8,10,10) - ({makanan_input:.1f}-6)/(8-6) = {makanan['Enak']:.2f}")

print(f"\nActive rules: R8 = min({makanan['Enak']:.2f}, {pelayanan['Biasa']:.1f}) = {evaluasi[7][1]:.1f}")
print(f"Output = ({evaluasi[7][1]:.1f}×18)/({evaluasi[7][1]:.1f}) = {tip:.2f}%")
print(f"Kategori: {kategori_tip(tip)} (15-20)")


----------------------------------------
TOLAK UKUR VALIDASI:
----------------------------------------
μ_Enak(7.5) = (7.5-6)/(8-6) = 0.75
μ_Biasa_pelayanan(4.0) = (4-3)/(5-3) = 0.50 WAIT - recalculate:
Actually: pelayanan=4.0, triangular(3,5,7): (4-3)/(5-3) = 0.50
Koreksi: μ_Biasa_pelayanan(4.0) = 0.50, bukan 1.0
R8: min(0.75, 0.50) = 0.50, R5: min(0, 0.50) = 0
Juga R4: min(0, 0) = 0, tapi pelayanan Buruk(4) = 0 karena 4>=4 - 0

Recalculate properly:
Pelayanan=4.0:
Buruk: trapezoidal(0,0,2,4) - (4-4)/(4-2) = 0.0 (x>=d - 0)
Biasa: triangular(3,5,7) - (4-3)/(5-3) = 0.5
Baik: trapezoidal(6,8,10,10) - 0

Makanan=7.5:
Buruk: trapezoidal(0,0,2,4) - 0
Biasa: triangular(3,5,7) - 7.5>=7 - 0
Enak: trapezoidal(6,8,10,10) - (7.5-6)/(8-6) = 0.75

Active rules: R8 = min(0.75, 0.5) = 0.5
Output = (0.5×18)/(0.5) = 18.00%
Kategori: Tinggi (15-20)
